# 03 · Preprocessing & Chunking

Clean the extracted Item 1A corpus and split it into model-ready **chunks** (≈ one risk
factor each). The chunking logic lives in `src/preprocess.py`; this notebook runs it and
performs quality-assurance checks.

Output: `data/processed/chunks_clean.csv`.

In [7]:
# Auto-reload edited src/ modules without restarting the kernel
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src import preprocess

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1 · Load inputs and build chunks

In [8]:
corpus, sector_map = preprocess.load_inputs()
print(f"Corpus: {len(corpus)} filings, {corpus['cik'].nunique()} companies, "
      f"{corpus['year'].min()}-{corpus['year'].max()}")

chunks = preprocess.build_chunks(corpus, sector_map)

OUT = preprocess.OUTPUT_CSV
OUT.parent.mkdir(parents=True, exist_ok=True)
chunks.to_csv(OUT, index=False)
print(f"Built {len(chunks)} chunks -> {OUT.relative_to(preprocess.DATA_DIR.parent)}")

Corpus: 849 filings, 55 companies, 2010-2025
Built 37453 chunks -> data/processed/chunks_clean.csv


## 2 · Overview

In [9]:
print(f"Chunks          : {len(chunks):,}")
print(f"Filings         : {len(corpus):,}")
print(f"Mean chunks/doc : {len(chunks)/len(corpus):.1f}")
print(f"Companies       : {chunks['cik'].nunique()}")
print(f"Sectors         : {chunks['sector'].nunique()}")
print("\nChunk word-count distribution:")
print(chunks['n_words'].describe().round(1).to_string())

Chunks          : 37,453
Filings         : 849
Mean chunks/doc : 44.1
Companies       : 55
Sectors         : 11

Chunk word-count distribution:
count    37453.0
mean       187.1
std        141.8
min         20.0
25%         57.0
50%        137.0
75%        365.0
max        657.0


## 3 · Quality assurance

These should all pass before topic modelling: no empty text, no out-of-range chunk
lengths, no missing metadata, and no chunk starting mid-sentence.

In [10]:
import re
problems = {}
problems['empty text']            = int((chunks['text'].str.strip() == '').sum())
problems['below MIN_CHUNK_WORDS'] = int((chunks['n_words'] < preprocess.MIN_CHUNK_WORDS).sum())
problems['above MAX_CHUNK_WORDS'] = int((chunks['n_words'] > preprocess.MAX_CHUNK_WORDS).sum())
problems['missing sector']        = int(chunks['sector'].isna().sum())
problems['starts lowercase/punct']= int((~chunks['text'].str.match(r'^[A-Z0-9"]')).sum())

for k, v in problems.items():
    flag = 'OK' if v == 0 else 'CHECK'
    print(f"  [{flag}] {k}: {v}")

  [OK] empty text: 0
  [OK] below MIN_CHUNK_WORDS: 0
  [CHECK] above MAX_CHUNK_WORDS: 50
  [OK] missing sector: 0
  [CHECK] starts lowercase/punct: 2174


## 4 · Chunks per year and per sector

Check temporal density (each year is a time-bin for BERTopic's topics-over-time) and that
no sector is missing. 2026 is expected to be thinner (partial filing cohort).

In [11]:
print('Chunks per year:')
print(chunks.groupby('year').size().to_string())
print('\nChunks per sector:')
print(chunks.groupby('sector').size().sort_values(ascending=False).to_string())

Chunks per year:
year
2010    3236
2011    3291
2012    4024
2013    3356
2014    2392
2015    2237
2016    2017
2017    2003
2018    2024
2019    1994
2020    1756
2021    1753
2022    1762
2023    1761
2024    1899
2025    1948

Chunks per sector:
sector
Financials                5196
Real Estate               4878
Utilities                 4689
Health Care               4645
Information Technology    4176
Consumer Discretionary    2982
Materials                 2669
Communication Services    2444
Industrials               2099
Energy                    1922
Consumer Staples          1753


## 5 · Inspect sample chunks

In [12]:
for _, r in chunks.sample(3, random_state=0).iterrows():
    print(f"\n=== {r['ticker']} {r['year']} · chunk {r['chunk_id']} · {r['sector']} ({r['n_words']}w) ===")
    print('TEXT:', r['text'][:300])


=== DLR 2013 · chunk 24 · Real Estate (20w) ===
TEXT: failure of contractors to perform on a timely basis or at all, or other misconduct on the part of contractors;

=== WELL 2018 · chunk 24 · Real Estate (195w) ===
TEXT: We maintain or require our tenants, operators and managers to maintain comprehensive insurance coverage on our properties and their operations with terms, conditions, limits and deductibles that we believe are customary for similarly-situated companies in our industry, and we frequently review our i

=== WBD 2010 · chunk 38 · Communication Services (30w) ===
TEXT: The personal educational media, lifelong learning, and travel industry investments by John S. Hendricks, a common stock director and our Founder, may conflict with or compete with our business activities.


---
Next: run BERTopic on filtered_corpus.csv (`src/model.py`).